# Daily forecast of financial time series group questions

## Forecast

In [ ]:
import matplotlib.pylab as plt
from get_market_pulse_25q3_questions import get_market_pulse_25q3_questions
from get_underlying_urls import get_underlying_urls
from urls_to_ticker import urls_to_ticker
from pprint import pprint
from get_data_for import get_data_for
from get_forward_period_start_and_end_date import get_forward_period_start_and_end_date
from get_observable import get_observable
from get_period_type import get_period_type
import pandas as pd
from forecast_high_on_any_day_in_forward_period import forecast_high_on_any_day_in_forward_period
from forecast_close_on_last_day_of_period import forecast_close_on_last_day_of_period
from forecast_stock_price_return_spread import forecast_stock_price_return_spread
from forecast_futures_total_price_return_spread import forecast_futures_total_price_return_spread
from forecast_quarter_diluted_eps_on_next_filing_date import forecast_quarter_diluted_eps_on_next_filing_date
from metaculus_generate_continuous_cdf import metaculus_generate_continuous_cdf
from standardize_cdf import standardize_cdf
from post_group_forecast import post_group_forecast
from forecast_general_binary import forecast_general_binary
from ensure_min_increase import ensure_min_increase
import os

pd.set_option('display.max_colwidth', 1000)

# Get open questions
ifps = get_market_pulse_25q3_questions()
results = get_underlying_urls(ifps)
results = {int(key): value for key, value in results.items()}
group_id_to_data = {int(x): results[x] for x in results}

# Acquire question data
sources = []
for key, (title, urls) in results.items():
    sources.extend(urls_to_ticker(urls))

sources = list(sorted(set(sources)))
history = {(x,y,z): get_data_for(x,y,z) for x,y,z in sources}

In [ ]:
for i, ifp in enumerate(ifps):
    # Organize data
    ifp['sources'] = urls_to_ticker(results[ifp['group']['id']][1])

In [ ]:
for i, ifp in enumerate(ifps):
    ifp['data'] = {x: history[x] for x in ifp['sources']}

In [ ]:
for i, ifp in enumerate(ifps):
    if 'following companies' in ifp['title']:
        company = ifp['title'].split('(')[1].split(')')[0]
        ifp['sources'] = [(x,y,z) for x,y,z in ifp['sources'] if y == company]
        ifp['data'] = {(x,y,z):w for (x,y,z),w in ifp['data'].items() if y == company}

In [ ]:
ifp['sources']

In [ ]:
for i, ifp in enumerate(ifps):
    # Get forward period start and end date
    if 'period' not in ifp or not ifp['period']:
        ifp['period'] = get_forward_period_start_and_end_date(ifp)

In [ ]:
for i, ifp in enumerate(ifps):
    # Parse observable of data
    if 'observable' not in ifp:
        ifp['observable'] = get_observable(ifp)

In [ ]:
for i, ifp in enumerate(ifps):
    if 'period_type' not in ifp:
        ifp['period_type'] = get_period_type(ifp)

In [ ]:
for i, ifp in enumerate(ifps):
    fn = f'glimt/forecast/{ifp["id"]}.json'
    if os.path.exists(fn):
          continue
    # Generate forecast for each question
    print(f"[{i}] {ifp['id']} {ifp['title']}")
    ifp['forecast'] = """Statistical analysis."""
    ifp['question_type'] = 'numeric'
    ### High on any day in forward biweekly period from today
    (observable, period_type) = (ifp['observable'], ifp['period_type'])
    if observable == 'High' and period_type == 'any day in biweekly period':
        print("******forecast_high_on_any_day_in_forward_period")
        rng, prediction = forecast_high_on_any_day_in_forward_period(ifp)
    ### Close on last day of biweekly period
    elif observable == 'Close' and period_type == 'last day of biweekly period':
        print("******forecast_close_on_last_day_of_period")
        rng, prediction = forecast_close_on_last_day_of_period(ifp)
    ### period stock price return on last vs first day of biweekly period
    elif (observable, period_type) == ('stock price Close return', 'return on last vs first day of biweekly period'):
        print("*** forecast_stock_price_return_spread")
        rng, prediction = forecast_stock_price_return_spread(ifp)
    ### period futures total price return on last vs first day of biweekly period
    elif (observable, period_type) == ('futures total price Close return', 'return on last vs first day of biweekly period'):
        print("*******forecast_futures_total_price_return_spread")
        rng, prediction = forecast_futures_total_price_return_spread(ifp)
    ### quarter diluted eps on next SEC filing date	
    elif (observable, period_type) == ('quarter diluted eps', 'next SEC filing date'):
        print("********forecast_quarter_diluted_eps_on_next_filing_date")
        rng, prediction = forecast_quarter_diluted_eps_on_next_filing_date(ifp)
    ### quarter total revenue on next SEC filing date	
    elif (observable, period_type)  == ('quarter total revenue', 'next SEC filing date'):
        print("*********forecast_quarter_diluted_eps_on_next_filing_date")
        rng, prediction = forecast_quarter_diluted_eps_on_next_filing_date(ifp)
    elif ifp['title'].startswith('Will '):
        print("BINARY")
        forecast_general_binary(ifp)
        continue
    else:
        raise Exception(f"Unhandled question [{ifp['id']}] {ifp['title']} {observable} {period_type}")
    ifp['prediction'] = prediction

In [ ]:
from smooth_percentiles import smooth_percentiles

for i, ifp in enumerate(ifps):   
    try:
        if type(ifp['prediction']) == float:
            ifp['question_type'] = 'binary'
    except:
        print('missing', ifp['id'])
    # Submit the question
    row = pd.Series()
    row['id_of_question'] = ifp['id']
    row['id_of_post'] = ifp['post_id']
    row['question_type'] = ifp['question_type']
    row['forecast'] = ifp['forecast']
    if ifp['question_type'] == 'numeric':
        p1 = ifp['prediction']
        p1[0] = ifp['scaling']['range_min']
        p1[-1] = ifp['scaling']['range_max']
        p1 = ensure_min_increase(p1)
        p2 = dict(zip(rng,p1))
        p4 = metaculus_generate_continuous_cdf(p2, ifp)
        p5 = standardize_cdf(p4, ifp['scaling'])
        row['prediction'] = p5
    elif ifp['question_type'] == 'binary':
        continue # already posted
    else:
        raise Exception(ifp['question_type'])
    try:
        post_group_forecast(row)
    except Exception as e:
        print("An error occurred:", e)

## Crowd alignment

In [48]:
import requests

def get_market_pulse_25q3_questions():
    url = 'https://www.metaculus.com/api/posts/'

    params = {
        'tournaments': 'market-pulse-25q3',
        'statuses': 'open',
        'with_cp': 'true',
        'include_cp_history': 'true',
        'include_descriptions': 'true',
        'order_by': '-published_at'
    }
    
    headers = {
        'accept': 'application/json'
    }
    
    response = requests.get(url, params=params, headers=headers)
    print(response.status_code)
    js = response.json()
    groups = js['results']
    ifps = []
    for x in groups:
        if 'group_of_questions' not in x:
            continue
        questions = x['group_of_questions']['questions']
        for question in questions:
            question['group'] = x
            ifps.append(question)

    ifps_sorted = [x[2] for x in list(sorted([(ifp['post_id'], ifp['id'], ifp) for ifp in ifps]))]
    ifps_sorted = [x for x in ifps_sorted if x['status'] == 'open']

    return ifps_sorted

In [49]:
ifps = get_market_pulse_25q3_questions()

200


In [52]:
ifps[0]['aggregations'].keys()

dict_keys(['recency_weighted', 'unweighted', 'single_aggregation', 'metaculus_prediction'])